In [ ]:
"""
Comparing BDEs from observations vs BDEs from associated dark observations. 
"""

In [ ]:
import numpy as np
from astropy.io import fits

#obs = "m3g20090617t045633" 
obs = "m3t20090630t124550"
path = f"/home/bekah/m3-pipeline-dev/data/l0_T/{obs}_l0.fits"

import numpy as np
from astropy.io import fits

with fits.open(path) as hdul:
    image = hdul[0].data

# don't use first 100 and last 100 or so frames for std dev? 
# I think this could be based on spacecraft geometry too... when does it start seeing a good amount of light? 
std_array = np.std(image.transpose(1, 0, 2)[100:-100, :, :], axis=0)


rows_to_ignore = [] #[49, 12] # global ones 
cols_to_ignore = [] #[319] 

ignore_mask = np.zeros(std_array.shape, dtype=bool)
ignore_mask[rows_to_ignore, :] = True
ignore_mask[:, cols_to_ignore] = True


def compute_local_std_neighborhood(std_array, ignore_mask, window_radius=2):
    """
    std and mean of surrounding pixels. could use median?
    """
    height, width = std_array.shape
    local_std_array = np.zeros_like(std_array, dtype=float)
    local_mean_array = np.zeros_like(std_array, dtype=float)
    
    for i in range(height):
        for j in range(width):
            i_min = max(0, i - window_radius)
            i_max = min(height, i + window_radius + 1)
            j_min = max(0, j - window_radius)
            j_max = min(width, j + window_radius + 1)
            
            neighborhood = std_array[i_min:i_max, j_min:j_max]
            neighborhood_mask = ignore_mask[i_min:i_max, j_min:j_max]
            valid_values = neighborhood[~neighborhood_mask]
            
            if len(valid_values) > 1:
                local_std_array[i, j] = np.std(valid_values)
                local_mean_array[i, j] = np.mean(valid_values)
            else:
                local_std_array[i, j] = np.nan
                local_mean_array[i, j] = np.nan
    
    return local_std_array, local_mean_array

local_std, local_mean = compute_local_std_neighborhood(
    std_array, 
    ignore_mask, 
    window_radius=2
)

threshold_sigma = 2 # 1.5 is NOT good 
deviation_from_local_mean = (std_array - local_mean) / local_std
outlier_mask = np.abs(deviation_from_local_mean) > threshold_sigma

outlier_mask[ignore_mask] = False

std_array_flagged = std_array.copy()

fits.writeto(
    f"/home/bekah/m3-pipeline-dev/data/l0_std/{obs}_std_flagged_l0.fits",
    outlier_mask.astype(int),
    overwrite=True
)

fits.writeto(
    f"/home/bekah/m3-pipeline-dev/data/l0_std/{obs}_std_image_l0.fits",
    std_array_flagged,
    overwrite=True
)

